# ResNet-18 on an 8 x 8 systolic array
This canonical human demo runs the exported ResNet-18 using the 8 x 8 NPU overlay (64 processing elements). It deliberately exposes every acceptance boundary instead of hiding the run behind a script. Open it from a deployed release on the PYNQ-Z1, select the board's PYNQ Python kernel, and execute one step at a time.

## 1. Locate the deployed release
Confirm that this kernel runs on Linux and that the notebook belongs to a release produced by `deploy_release.ps1`.

In [ ]:
from datetime import datetime, timezone
import hashlib
import json
import os
from pathlib import Path
import sys

import numpy as np

if not sys.platform.startswith('linux'):
    raise RuntimeError('Open this notebook on the PYNQ-Z1, not Windows')

start = Path.cwd().resolve()
release_root = next(
    (candidate for candidate in (start, *start.parents)
     if (candidate / 'deployment.json').is_file()
     and (candidate / 'examples/resnet18/model').is_dir()
     and (candidate / 'build/vivado/npu_matrix_8x8/artifacts').is_dir()),
    None,
)
if release_root is None:
    raise RuntimeError('Open this notebook from a deployed ResNet-18 release')

model_dir = release_root / 'examples/resnet18/model'
artifact_dir = release_root / 'build/vivado/npu_matrix_8x8/artifacts'
print(f'Release root: {release_root}')
print(f'Python: {sys.executable}')

## 2. Inspect deployment provenance
Read the commits recorded during transfer. Do not continue if these are not the intended source and overlay revisions.

In [ ]:
deployment = json.loads((release_root / 'deployment.json').read_text(encoding='utf-8'))
assert deployment['magic'] == 'NPU_RESNET18_DEPLOYMENT'
assert deployment['format'] == {'major': 1, 'minor': 0}
assert len(deployment['artifact_source_commit']) == 40
assert len(deployment['deployed_source_commit']) == 40
source_mismatch = (
    deployment['artifact_source_commit']
    != deployment['deployed_source_commit']
)
if source_mismatch and not deployment['allow_source_mismatch']:
    raise RuntimeError('Artifact and deployed commits differ without approval')
print(json.dumps(deployment, indent=2, sort_keys=True))
print('Evidence class:', 'development' if source_mismatch else 'trusted')

## 3. Verify every model input
Validate the pinned checkpoint, conversion record, host acceptance, model manifest, payload, and validation tensor. Display each accepted file with its byte count and SHA-256 digest.

In [ ]:
sys.path.insert(0, str(release_root))
from examples.resnet18.package_example import validate_workspace

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

source_metadata_path = release_root / 'examples/resnet18/model-source.json'
source_metadata = json.loads(source_metadata_path.read_text(encoding='utf-8'))
validated_assets = validate_workspace(model_dir, source_metadata_path)
reviewed_assets = [
    source_metadata_path, model_dir / source_metadata['filename'], *validated_assets
]
model_files = [
    {
        'file': str(path.relative_to(release_root)),
        'bytes': path.stat().st_size,
        'sha256': file_sha256(path),
    }
    for path in reviewed_assets
]
model_files

## 4. Verify the BIT/HWH overlay
Check artifact digests, metadata, target part, and provenance before programming the FPGA. The manifest source must match the commit recorded at deployment.

In [ ]:
from src.runtime.verify_overlay import verify_artifacts

overlay_record = verify_artifacts(artifact_dir)
assert overlay_record.get('array_size') == 8, 'This ResNet demo requires the 8 x 8 overlay'
assert overlay_record['source_commit'].lower() == deployment['artifact_source_commit']
overlay_summary = {
    'array_size': overlay_record['array_size'],
    'target_part': overlay_record['target_part'],
    'source_commit': overlay_record['source_commit'],
    'bit_sha256': overlay_record['bit']['sha256'],
    'hwh_sha256': overlay_record['hwh']['sha256'],
}
overlay_summary

## 5. Reconstruct and inspect the exported model
Reload the serialized model through the production loader, which checks its payload, graph, memory plan, accumulator certificates, ABI, and capabilities. Then inspect the input and graph dimensions.

In [ ]:
from src.runtime import load_model_package

model = load_model_package(model_dir / 'resnet18.npu.json')
validation_input = np.load(model_dir / 'resnet18.validation.npy', allow_pickle=False)
host_acceptance = json.loads((model_dir / 'acceptance.json').read_text(encoding='utf-8'))
model_summary = {
    'input_name': model.graph.inputs[0],
    'input_shape': list(validation_input.shape),
    'input_dtype': str(validation_input.dtype),
    'commands': len(model.graph.commands),
    'outputs': list(model.graph.outputs),
    'arena_bytes': model.memory_plan.arena_bytes,
    'required_abi_major': model.required_abi_major,
    'required_capabilities': f'0x{model.required_capabilities:08x}',
}
model_summary

## 6. Program the FPGA and prove the runtime is physical
This is the first hardware-changing step. It loads the BIT file, discovers the accelerator and DMA from HWH metadata, and negotiates the hardware ABI.

In [ ]:
os.environ['XILINX_XRT'] = '/usr'
from src.runtime import NPURuntime, load_pynq_runtime

physical = load_pynq_runtime(artifact_dir / 'npu_matrix.bit')
assert isinstance(physical, NPURuntime), type(physical)
assert (physical.max_m, physical.max_n, physical.max_k) == (8, 8, 256), 'Expected 8 x 8 hardware with MAX_K=256'
hardware_summary = {
    'systolic_array': '8 x 8',
    'processing_elements': 64,
    'runtime_class': type(physical).__name__,
    'abi_major': physical.abi_major,
    'capabilities': f'0x{physical.capabilities:08x}',
    'matrix_limits': [physical.max_m, physical.max_n, physical.max_k],
}
hardware_summary

## 7. Execute ResNet-18 on the NPU
Run the validated ResNet-18 input through the production model runtime. Convolution and fully-connected operations use 8 x 8 matrix tiles; other graph operations retain their existing software implementation. Inspect MAC count, physical job count and elapsed time before considering the output.

In [ ]:
from src.runtime import NPUModelRuntime
import time

inference_started = time.monotonic()
result = NPUModelRuntime(physical, model).run(
    {'input': validation_input}, software_timeout=86400.0
)
runtime_summary = {
    'systolic_array': '8 x 8',
    'elapsed_seconds': time.monotonic() - inference_started,
    'mac_count': result.metrics.mac_count,
    'physical_jobs': result.metrics.physical_jobs,
}
assert result.metrics.physical_jobs > 0
runtime_summary

## 8. Compare every output with independent host evidence
Compute each physical output digest here and compare it with the prior host acceptance. Every row must match.

In [ ]:
def array_sha256(value):
    array = np.ascontiguousarray(value)
    digest = hashlib.sha256()
    digest.update(str(array.dtype).encode('ascii'))
    digest.update(b'\0')
    digest.update(json.dumps(list(array.shape), separators=(',', ':')).encode('ascii'))
    digest.update(b'\0')
    digest.update(array.tobytes(order='C'))
    return digest.hexdigest()

capture_review = []
for name in model.graph.outputs:
    actual = array_sha256(result.outputs[name])
    expected = host_acceptance['captures'][name]['sha256']
    capture_review.append({
        'name': name,
        'shape': list(result.outputs[name].shape),
        'expected_sha256': expected,
        'actual_sha256': actual,
        'match': actual == expected,
    })
assert capture_review and all(row['match'] for row in capture_review)
capture_review

## 9. Review the complete evidence before publication
This cell assembles but does not write evidence. Inspect every field, especially evidence type, commits, hashes, ABI, capabilities, MAC count, and physical job count.

In [ ]:
evidence_type = 'physical-pynq-z1-development' if source_mismatch else 'physical-pynq-z1'
evidence = {
    'captures': {
        row['name']: {'sha256': row['actual_sha256'], 'shape': row['shape']}
        for row in capture_review
    },
    'evidence_type': evidence_type,
    'format': {'major': 1, 'minor': 0},
    'host_acceptance_sha256': file_sha256(model_dir / 'acceptance.json'),
    'magic': 'NPU_RESNET18_BOARD_ACCEPTANCE',
    'model_manifest_sha256': file_sha256(model_dir / 'resnet18.npu.json'),
    'overlay': {
        'bit_sha256': overlay_record['bit']['sha256'],
        'deployed_source_commit': deployment['deployed_source_commit'],
        'hwh_sha256': overlay_record['hwh']['sha256'],
        'source_commit': overlay_record['source_commit'],
        'source_mismatch_allowed': source_mismatch,
        'array_size': overlay_record['array_size'],
        'target_part': overlay_record['target_part'],
    },
    'result': 'pass',
    'runtime': {
        'abi_major': physical.abi_major,
        'capabilities': physical.capabilities,
        'mac_count': result.metrics.mac_count,
        'physical_jobs': result.metrics.physical_jobs,
        'physical_limits': [physical.max_m, physical.max_n, physical.max_k],
    },
}
evidence

## 10. Human approval and evidence write
Do not change `human_approves` until you have reviewed Steps 2 through 9. Setting it to `True` is the explicit human acceptance action.

In [ ]:
human_approves = False  # Change to True only after reviewing all evidence above.
if not human_approves:
    raise RuntimeError('Human approval is required before writing PASS evidence')

stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
evidence_path = release_root / f'notebook-evidence-{stamp}.json'
payload = json.dumps(
    evidence, allow_nan=False, sort_keys=True, separators=(',', ':')
) + '\n'
with evidence_path.open('x', encoding='utf-8', newline='\n') as stream:
    stream.write(payload)
    stream.flush()
    os.fsync(stream.fileno())
print(f'PASS [{evidence_type}]: human-reviewed notebook demo')
print(f'Evidence: {evidence_path}')